In [1]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, emb_size=768, img_size=224):
        super().__init__()
        self.patch_size = patch_size
        
        # 1. Project image patches to an embedding dimension
        # Using Conv2d is an efficient trick to extract non-overlapping patches 
        # and project them to the emb_size all at once.
        self.projection = nn.Conv2d(
            in_channels, 
            emb_size, 
            kernel_size=patch_size, 
            stride=patch_size
        )
        
        # 2. Learnable Classification Token ([CLS])
        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_size))
        
        # 3. Learnable Positional Embeddings
        # Calculate total patches: (224 / 16) * (224 / 16) = 196 patches
        num_patches = (img_size // patch_size) ** 2 
        # +1 is for the [CLS] token
        self.positions = nn.Parameter(torch.randn(1, num_patches + 1, emb_size))
        
    def forward(self, x):
        b, c, h, w = x.shape
        
        # Project and flatten
        x = self.projection(x)       # Shape: (Batch, emb_size, H/P, W/P)
        x = x.flatten(2)             # Shape: (Batch, emb_size, Num_patches)
        x = x.transpose(1, 2)        # Shape: (Batch, Num_patches, emb_size)
        
        # Expand the CLS token for the whole batch
        cls_tokens = self.cls_token.expand(b, -1, -1) # Shape: (Batch, 1, emb_size)
        
        # Prepend CLS token to the patches
        x = torch.cat((cls_tokens, x), dim=1)         # Shape: (Batch, Num_patches + 1, emb_size)
        
        # Add position embeddings
        x += self.positions
        
        return x

In [2]:
# Create a dummy image batch: Batch size 8, 3 Color Channels, 224x224 pixels
dummy_images = torch.randn(8, 3, 224, 224)

# Initialize the module
patch_embedder = PatchEmbedding()

# Pass the images through
embedded_patches = patch_embedder(dummy_images)

print(f"Input shape: {dummy_images.shape}")
print(f"Output shape: {embedded_patches.shape}") 
# Expected Output: torch.Size([8, 197, 768]) 
# (8 images, 196 patches + 1 CLS token, 768 embedding dimension)

Input shape: torch.Size([8, 3, 224, 224])
Output shape: torch.Size([8, 197, 768])


In [3]:
import torch.nn as nn

class MLP(nn.Module):
    """The green block in your diagram."""
    def __init__(self, emb_size, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        # In ViTs, the hidden layer is usually 4x larger than the embedding size
        hidden_size = int(emb_size * mlp_ratio) 
        
        self.network = nn.Sequential(
            nn.Linear(emb_size, hidden_size),
            nn.GELU(), # Standard activation function for Transformers
            nn.Dropout(dropout),
            nn.Linear(hidden_size, emb_size),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)

class TransformerEncoderBlock(nn.Module):
    """The entire gray box in your diagram."""
    def __init__(self, emb_size=768, num_heads=12, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        
        # 1. First Norm Layer (orange box)
        self.norm1 = nn.LayerNorm(emb_size)
        
        # 2. Multi-Head Attention (blue box)
        # batch_first=True because our data is (Batch, Patches, Embedding)
        self.attn = nn.MultiheadAttention(
            embed_dim=emb_size, 
            num_heads=num_heads, 
            dropout=dropout, 
            batch_first=True
        )
        
        # 3. Second Norm Layer (orange box)
        self.norm2 = nn.LayerNorm(emb_size)
        
        # 4. MLP Layer (green box)
        self.mlp = MLP(emb_size, mlp_ratio, dropout)

    def forward(self, x):
        # --- First Residual Block ---
        # Normalize the input
        norm_x = self.norm1(x)
        # Apply Multi-Head Attention. It returns (output, weights), we just want the output.
        attn_output, _ = self.attn(norm_x, norm_x, norm_x)
        # The first '+' circle in your diagram: Add original input to attention output
        x = x + attn_output
        
        # --- Second Residual Block ---
        # Normalize the new x
        norm_x = self.norm2(x)
        # Pass through MLP
        mlp_output = self.mlp(norm_x)
        # The second '+' circle in your diagram: Add x to the MLP output
        x = x + mlp_output
        
        return x

In [4]:
class VisionTransformer(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, emb_size=768, img_size=224, 
                 num_blocks=12, num_heads=12, num_classes=1000):
        super().__init__()
        
        # 1. Patch Embedding (from Step 1)
        self.patch_embed = PatchEmbedding(in_channels, patch_size, emb_size, img_size)
        
        # 2. Stack of Transformer Encoders (from Step 2)
        # We use nn.ModuleList to hold multiple blocks
        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(emb_size, num_heads) 
            for _ in range(num_blocks)
        ])
        
        # 3. Final Layer Normalization
        # It's standard practice to add one last normalization before the classification head
        self.norm = nn.LayerNorm(emb_size)
        
        # 4. Classification Head
        # Maps the embedding size (768) to the number of classes we want to predict
        self.head = nn.Linear(emb_size, num_classes)
        
    def forward(self, x):
        # 1. Turn image into embedded patches with [CLS] token and positional encoding
        x = self.patch_embed(x)
        
        # 2. Pass the patches through every encoder block sequentially
        for block in self.encoder_blocks:
            x = block(x)
            
        # 3. Extract the [CLS] token
        # x shape is (Batch, 197, 768). 
        # We want the 0th token for every item in the batch: x[:, 0]
        cls_token_final = x[:, 0]
        
        # 4. Normalize and classify
        out = self.norm(cls_token_final)
        out = self.head(out)
        
        return out

In [5]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Use GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Prepare Data Transformations
# We add some basic augmentation (cropping/flipping) to help the model learn better
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    #transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# 2. Download and Load CIFAR-10
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)

# Batch size of 128 is a safe sweet spot for 8GB VRAM
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

Using device: cuda


In [6]:
# Create a scaled-down ViT for 32x32 images
mini_vit = VisionTransformer(
    in_channels=3, 
    patch_size=4,       # 32 / 4 = 8 patches per side (64 patches total)
    emb_size=256,       # Scaled down from 768, 256
    img_size=32,        # Native CIFAR-10 size
    num_blocks=8,       # Half the depth of a standard ViT
    num_heads=8,        # 256 is divisible by 8, 8
    num_classes=10      # 10 categories in CIFAR-10
).to(device) # Move the model to your GPU!

# Check total parameters
total_params = sum(p.numel() for p in mini_vit.parameters())
print(f"Total parameters: {total_params:,}") 
# This should be around 4-5 million parameters (very manageable!)

Total parameters: 6,350,602


In [7]:
import torch.optim as optim
import torch.nn as nn
import time

# CrossEntropy is standard for multi-class classification
criterion = nn.CrossEntropyLoss()
# AdamW is generally preferred over standard Adam for Transformers
optimizer = optim.AdamW(mini_vit.parameters(), lr=0.001, weight_decay=0.05)

epochs = 10 # Start small, just to see it work
#scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
print("Starting training 'The Hard Way'...")
for epoch in range(epochs):
    mini_vit.train()
    running_loss = 0.0
    start_time = time.time()
    
    for i, data in enumerate(trainloader, 0):
        # 1. Get the inputs and labels, and move them to the GPU
        inputs, labels = data[0].to(device), data[1].to(device)

        # 2. Zero the parameter gradients
        optimizer.zero_grad()

        # 3. Forward pass (make predictions)
        outputs = mini_vit(inputs)
        
        # 4. Calculate Loss
        loss = criterion(outputs, labels)
        
        # 5. Backward pass (calculate gradients)
        loss.backward()
        
        # 6. Optimize (update weights)
        optimizer.step()

        running_loss += loss.item()
        
        # Print progress every 100 batches
        if i % 100 == 99:
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}")
            running_loss = 0.0
            
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1} finished in {epoch_time:.2f} seconds.")
    #scheduler.step()

print("Finished Training!")

Starting training 'The Hard Way'...
[1,   100] loss: 2.154
[1,   200] loss: 1.945
[1,   300] loss: 1.849
Epoch 1 finished in 30.48 seconds.
[2,   100] loss: 1.760
[2,   200] loss: 1.728
[2,   300] loss: 1.721
Epoch 2 finished in 30.57 seconds.
[3,   100] loss: 1.681
[3,   200] loss: 1.670
[3,   300] loss: 1.660
Epoch 3 finished in 30.79 seconds.
[4,   100] loss: 1.632
[4,   200] loss: 1.647
[4,   300] loss: 1.614
Epoch 4 finished in 31.43 seconds.
[5,   100] loss: 1.596
[5,   200] loss: 1.600
[5,   300] loss: 1.577
Epoch 5 finished in 31.67 seconds.
[6,   100] loss: 1.557
[6,   200] loss: 1.556
[6,   300] loss: 1.548
Epoch 6 finished in 31.74 seconds.
[7,   100] loss: 1.512
[7,   200] loss: 1.519
[7,   300] loss: 1.512
Epoch 7 finished in 31.83 seconds.
[8,   100] loss: 1.465
[8,   200] loss: 1.498
[8,   300] loss: 1.494
Epoch 8 finished in 31.89 seconds.
[9,   100] loss: 1.469
[9,   200] loss: 1.464
[9,   300] loss: 1.464
Epoch 9 finished in 31.89 seconds.
[10,   100] loss: 1.448
[10,

In [8]:
# 1. Prepare Test Transformations
# Important: We do NOT want to randomly crop or flip the test images!
# We just want to convert them to tensors and normalize them.
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# 2. Load the CIFAR-10 Test Set
# Notice we set train=False this time!
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)

# We don't need to shuffle the test data, it doesn't affect accuracy
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# 3. The Evaluation Loop
correct = 0
total = 0

print("Testing the Mini-ViT on unseen data...")

# Set the model to evaluation mode (turns off dropout)
mini_vit.eval()

# Turn off gradient calculation (torch.no_grad) because we aren't training.
# This saves VRAM and drastically speeds up the testing process.
with torch.no_grad():
    for data in testloader:
        inputs, labels = data[0].to(device), data[1].to(device)
        
        # Forward pass: make predictions
        outputs = mini_vit(inputs)
        
        # The output is a tensor of 10 probabilities. 
        # We grab the index of the highest probability as our prediction.
        _, predicted = torch.max(outputs.data, 1)
        
        # Tally up the totals
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# 4. Calculate Final Accuracy
accuracy = 100 * correct / total
print(f'Accuracy of the Mini-ViT on the 10,000 test images: {accuracy:.2f}%')

Testing the Mini-ViT on unseen data...
Accuracy of the Mini-ViT on the 10,000 test images: 51.76%
